# Prompt playground: Gemma on 1-2 sign crops

Try a prompt on a couple of crops and see the raw answer, the parsed entries and the time.
Uses the same image preparation and parser as `vlm.py`, so what you see here is what the
pipeline gets. Needs Ollama running with the model pulled (`ollama pull gemma4:e4b`).

Put crops in `notebooks/samples/` (png/jpg), or point `SAMPLES` at any image paths.

In [ ]:
import json, sys, time
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import cv2
import matplotlib.pyplot as plt
import ollama

import vlm                      # the pipeline's prompts, image prep and parser

MODEL = "gemma4:e4b"
client = ollama.Client()        # ollama.Client(host="http://localhost:11434")
print("prompts in vlm.py:", list(vlm.PROMPTS), "| default:", vlm.DEFAULT_PROMPT)

## Samples
One or two crops. Each entry: a path, and optionally what the labels say, to compare by eye.

In [ ]:
SAMPLES = {
    # name: (path, expected)  -- expected is just a note for you
    "S0":  (ROOT / "notebooks/samples/S0.png",  "paterson link: forward, event hall: forward"),
    "S10": (ROOT / "notebooks/samples/S10.png", "orchard mrt: up-right, concierge: up-right"),
}

def load(path):
    img = cv2.imread(str(path))
    if img is None:
        raise FileNotFoundError(path)
    return img

def show(img, title=""):
    plt.figure(figsize=(10, 3)); plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(title); plt.axis("off"); plt.show()

images = {}
for name, (path, note) in SAMPLES.items():
    try:
        images[name] = load(path)
        show(images[name], f"{name}  (expected: {note})")
    except FileNotFoundError as e:
        print("missing:", e)

## The call
`ask()` sends one image and one prompt, exactly like `vlm.SignReader` does (same resizing,
JPEG quality, temperature 0, thinking off). `json_mode=False` lets you ask free-form questions.

In [ ]:
def ask(img, prompt, json_mode=True, think=False, temperature=0.0, model=MODEL):
    ok, buf = cv2.imencode(".jpg", vlm._prep(img), [cv2.IMWRITE_JPEG_QUALITY, 92])
    t0 = time.time()
    resp = client.chat(model=model,
                       messages=[{"role": "user", "content": prompt, "images": [buf.tobytes()]}],
                       format="json" if json_mode else None,
                       options={"temperature": temperature}, think=think, keep_alive="2h")
    dt = time.time() - t0
    raw = resp["message"]["content"]
    return raw, dt

def run(prompt, names=None, json_mode=True, think=False, label="custom"):
    for name in names or images:
        raw, dt = ask(images[name], prompt, json_mode, think)
        print(f"=== {name}  [{label}]  {dt:.2f}s  (expected: {SAMPLES[name][1]})")
        if json_mode:
            for e in vlm.parse(raw, "v5"):
                extra = "".join(f"  {k}={v}" for k, v in e.items() if k not in ("text", "arrow"))
                print(f"  {e['text']!r:45} -> {e['arrow']}{extra}")
            print("  raw:", raw[:400])
        else:
            print(raw)
        print()

## Existing prompts (v4 = default, v5 = latest)

In [ ]:
run(vlm.PROMPTS["v4"], label="v4")
run(vlm.PROMPTS["v5"], label="v5")

## Your prompt
Edit and re-run. Keep the JSON shape `{"entries": [{"text", "arrow"}]}` (optionally `"from"`)
so `vlm.parse` reads it the way the pipeline would.

In [ ]:
MY_PROMPT = """Read this navigation sign (a cropped photo).
List every destination on it with the arrow that points the way to it, as seen facing the sign:
up, left, right, up-left, up-right, down, down-left, down-right, or none.
Return only JSON: {"entries": [{"text": "...", "arrow": "..."}]}"""

run(MY_PROMPT, label="mine")

## Small focused questions
One question at a time, free-form or JSON. Useful to see what the model actually perceives.

In [ ]:
run("Look only at the arrows on this sign. For each arrow, say where it is "
    "(which row, left or right end) and which way it points, using: up, left, right, "
    "up-left, up-right, down, down-left, down-right.", json_mode=False, label="arrows only")

## Zoom in: crop part of the sign
`region(img, x0, y0, x1, y1)` takes fractions of the width/height, e.g. the right-hand fifth
where the arrows are. Handy to test whether the model reads an isolated arrow correctly.

In [ ]:
def region(img, x0, y0, x1, y1):
    h, w = img.shape[:2]
    return img[int(y0 * h):int(y1 * h), int(x0 * w):int(x1 * w)]

name = "S10"
part = region(images[name], 0.75, 0.0, 0.92, 1.0)     # adjust to frame the arrows
show(part, f"{name}: region")
raw, dt = ask(part, "Which way does each arrow in this image point? Answer with one of: "
              "up, left, right, up-left, up-right, down, down-left, down-right.", json_mode=False)
print(f"{dt:.2f}s ->", raw)

## Mark a destination with a box (focused per-goal question)
Draw a box around one destination and ask only about it.

In [ ]:
def boxed(img, x0, y0, x1, y1, color=(0, 0, 255)):
    out = img.copy(); h, w = out.shape[:2]
    cv2.rectangle(out, (int(x0 * w), int(y0 * h)), (int(x1 * w), int(y1 * h)), color, 3)
    return out

name = "S0"
marked = boxed(images[name], 0.15, 0.18, 0.70, 0.45)  # adjust around one destination
show(marked, f"{name}: marked")
raw, dt = ask(marked, "Which arrow on this sign points the way to the destination inside the "
              "red box? Answer one of: up, left, right, up-left, up-right, down, down-left, "
              "down-right, none.", json_mode=False)
print(f"{dt:.2f}s ->", raw)